# Book Management

## Import Libraries

In [ ]:
import os
import pandas as pd
import re
import sqlite3
import tkinter as tk
from tkinter import messagebox
import ttkbootstrap as ttk
from ipynb.fs.full.base_frame import BaseFrame

## Book Entity

In [ ]:
class Book:
    """Represents a catalog book with bibliographic and inventory fields."""

    def __init__(self, title, author, isbn, year, quantity):
        """Initialize a book instance.

        Args:
            title: Display title used in UI, filtering and reports.
            author: Author name used in UI and filtering.
            isbn: ISBN identifier used as a unique business key.
            year: Publication year displayed in the catalog.
            quantity: Inventory quantity available for lending.
        """
        self.title = title
        self.author = author
        self.isbn = isbn
        self.year = year
        self.quantity = quantity

    def __str__(self):
        """Return a human-readable summary for quick inspection."""
        return f"Book: {self.title} by {self.author}, ISBN: {self.isbn}, Year: {self.year}, Quantity: {self.quantity}"

    def __repr__(self):
        """Return a precise representation for debugging and tests."""
        return f"Book(title={self.title!r}, author={self.author!r}, isbn={self.isbn!r}, year={self.year}, quantity={self.quantity})"

## Book Management Class

In [ ]:
# noinspection PyTypeChecker
class BookManagement(BaseFrame):
    """Manage book catalog data, filters, UI interactions, and persistence."""

    isbn_pattern = re.compile(r"^[0-9xX-]*$")

    def __init__(self, master=None, no_ui=False, shared_categories=None):
        """Initialize in-memory state and optional Tkinter/UI bindings.

        Args:
            master: Parent Tk widget used when rendering the feature.
            no_ui: If True, skip UI objects to support unit tests.
        """
        self.categories = shared_categories if shared_categories is not None else self._default_categories()

        if not no_ui:
            self.editing_original_isbn = None
            self.editing_original_category = None
            self.editing_row_index = None
            self.edit_title_var = tk.StringVar()
            self.edit_author_var = tk.StringVar()
            self.edit_isbn_var = tk.StringVar()
            self.edit_year_var = tk.StringVar()
            self.edit_quantity_var = tk.IntVar(value=0)
            self.edit_category_var = tk.StringVar(value="Fiction")
            self.filter_author_var = tk.StringVar()
            self.filter_title_var = tk.StringVar()
            self.filter_category_var = tk.StringVar(value="All")
            super().__init__(master)
            self.refresh_table_from_data()

    @staticmethod
    def _default_categories() -> dict:
        """Return the default category mapping with empty book lists."""
        return {
            "Fiction": [],
            "Science Fiction": [],
            "Fantasy": [],
            "Mystery": [],
            "Romance": [],
            "Technology": [],
            "Business": [],
            "Education": []
        }

    def add_book(self, category: str, book: Book):
        """Add a book to a category after validating ISBN uniqueness.

        Args:
            category: Destination category name in the category mapping.
            book: Book instance to add.

        Raises:
            ValueError: If an existing book already has the same ISBN.
        """
        if self.has_duplicate_isbn(book.isbn):
            raise ValueError("A book with this ISBN already exists.")
        self.categories[category].append(book)

    def has_duplicate_isbn(self, isbn: str, exclude_book: Book = None) -> bool:
        """
        Checks if an ISBN already exists in any category list.

        How it works:
        - Normalizes the incoming ISBN (strip spaces).
        - Iterates over all books in all categories.
        - Compares each stored ISBN with the normalized ISBN.
        - Returns True as soon as a duplicate is found; otherwise False.

        Why `exclude_book` is important:
        - During edit operations, the current book may keep the same ISBN.
        - Without `exclude_book`, the function would compare the book with itself
          and incorrectly mark it as duplicate.
        - `exclude_book` skips that specific object so only real duplicates are detected.
        - This is also important when moving a book between categories: the same
          book object is reallocated, and we must ignore it to avoid a false duplicate.

        Args:
            isbn: Candidate ISBN to be checked.
            exclude_book: Optional book object ignored during comparison.

        Returns:
            bool: True when another book with the same ISBN exists, else False.
        """
        normalized_isbn = (isbn or "").strip()
        if not normalized_isbn:
            return False

        for books in self.categories.values():
            for current_book in books:
                if not isinstance(current_book, Book):
                    continue
                if exclude_book is not None and current_book is exclude_book:
                    continue
                if str(current_book.isbn).strip() == normalized_isbn:
                    return True
        return False

    def filter_categories(self, category: str = "All", title: str = "", author: str = "") -> dict:
        """Filter categories using optional category, title, and author inputs.

        Args:
            category: "All" or a valid category name.
            title: Optional case-insensitive title fragment.
            author: Optional case-insensitive author fragment.

        Returns:
            dict: Category-to-books mapping containing only matching books.

        Raises:
            ValueError: If category is neither "All" nor an existing key.
        """
        selected_category = (category or "All").strip()
        title_filter = (title or "").strip().lower()
        author_filter = (author or "").strip().lower()

        categories_list = list(self.categories.keys())
        if selected_category.lower() == "all":
            target_categories = categories_list
        elif selected_category in self.categories:
            target_categories = [selected_category]
        else:
            raise ValueError("Category must be 'All' or an existing category.")

        filtered_categories = {name: [] for name in target_categories}

        for category_name in target_categories:
            for book in self.categories.get(category_name, []):
                if not isinstance(book, Book):
                    continue

                if title_filter and title_filter not in str(book.title).lower():
                    continue
                if author_filter and author_filter not in str(book.author).lower():
                    continue

                filtered_categories[category_name].append(book)

        return filtered_categories

    """
    The following functions are used for visual data presentation.
    They are used to display data in a more user-friendly way
    """

    # --- Base Frame UI Functions Overridden ---
    def get_title(self) -> str:
        """Return the feature title shown in the main panel."""

        return "📚 Book Management"

    def build_right_panel(self) -> None:
        """Build right-side action controls for save/load operations."""

        super().build_right_panel()

        actions_frame = ttk.Frame(master=self.right_panel, padding=(10, 8))
        actions_frame.pack(fill=tk.X)

        ttk.Button(
            master=actions_frame,
            text="Save Data",
            bootstyle="success",
            command=self.save_to_sqlite
        ).pack(fill=tk.X, pady=(0, 8))

        ttk.Button(
            master=actions_frame,
            text="Load Data",
            command=self.load_data
        ).pack(fill=tk.X)

    def build_left_panel(self) -> None:
        """Build form, filter, and data-table sections and bind edit events."""

        super().build_left_panel()
        self.build_form()
        self.build_filter()
        self.refresh_table_from_data()
        self.create_data_table(self.left_panel_container, row=0, column=1, rowspan=3,
                               colspan=2, sticky=tk.NSEW, padx=10, pady=(10,0),
                               title="Books", height=528)
        self.table.bind("<Double-Button-1>", lambda _event: self.edit_selected_row())

    def clear_edit_form(self) -> None:
        """Clear all edit fields and reset row-edit tracking attributes."""

        self.edit_title_var.set("")
        self.edit_author_var.set("")
        self.edit_isbn_var.set("")
        self.edit_year_var.set("")
        self.edit_quantity_var.set(0)
        self.edit_category_var.set("Fiction")
        self.editing_original_isbn = None
        self.editing_original_category = None
        self.editing_row_index = None

    # --- Validation Functions ---
    @staticmethod
    def is_valid_isbn_code(value: str) -> bool:
        """Validate ISBN characters and enforce maximum normalized length."""
        if value is None:
            return False

        clean_value = value.replace("-", "")
        if len(clean_value) > 13:
            return False

        return re.search(BookManagement.isbn_pattern, clean_value) is not None

    @staticmethod
    def validate_isbn_field(value: str) -> bool:
        """Validate ISBN entry input during key-by-key UI typing."""

        if value == "":
            return True
        return BookManagement.is_valid_isbn_code(value)

    @staticmethod
    def validate_quantity_field(value: str) -> bool:
        """Validate quantity input as an integer in the range 0..1000."""

        if value == "":
            return True
        if not value.isdigit():
            return False
        return 0 <= int(value) <= 1000

    # --- Forms, Save Data and Editin Functions ---
    def save_changes(self) -> None:
        """Create a new book or update an existing one from form values.

        This method validates fields, enforces ISBN uniqueness, updates
        in-memory structures, and refreshes table content/selection.
        """

        title = self.edit_title_var.get().strip()
        author = self.edit_author_var.get().strip()
        isbn = self.edit_isbn_var.get().strip()
        year_value = self.edit_year_var.get().strip()
        category = self.edit_category_var.get().strip()

        # Required text fields
        if not title:
            messagebox.showerror("Validation Error", "Title is required.")
            return
        if not author:
            messagebox.showerror("Validation Error", "Author is required.")
            return

        # ISBN validation
        if not isbn:
            messagebox.showerror("Validation Error", "ISBN is required.")
            return
        if not BookManagement.is_valid_isbn_code(isbn):
            messagebox.showerror("Validation Error", "Invalid ISBN format.")
            return

        # Year validation
        if not year_value:
            messagebox.showerror("Validation Error", "Year is required.")
            return
        if not year_value.isdigit():
            messagebox.showerror("Validation Error", "Year must be an integer.")
            return
        year = int(year_value)

        # Quantity validation
        quantity_value = str(self.edit_quantity_var.get()).strip()
        if not BookManagement.validate_quantity_field(quantity_value):
            messagebox.showerror("Validation Error", "Quantity must be an integer between 0 and 1000.")
            return
        quantity = int(quantity_value)

        # Category validation
        if not category:
            messagebox.showerror("Validation Error", "Category is required.")
            return
        if category not in self.categories:
            self.categories[category] = []

        is_editing = (
            self.editing_original_isbn is not None
            and self.editing_original_category is not None
            and self.editing_row_index is not None
        )

        if is_editing:
            original_books = self.categories.get(self.editing_original_category, [])
            original_index = None

            for index, current_book in enumerate(original_books):
                if isinstance(current_book, Book) and current_book.isbn == self.editing_original_isbn:
                    original_index = index
                    break

            if original_index is None:
                messagebox.showerror("Update Error", "The selected row could not be found.")
                self.refresh_table_from_data()
                self.clear_edit_form()
                return

            book = original_books[original_index]
            if self.has_duplicate_isbn(isbn, exclude_book=book):
                messagebox.showerror("Validation Error", "A book with this ISBN already exists.")
                return

            book.title = title
            book.author = author
            book.isbn = isbn
            book.year = year
            book.quantity = quantity
            if category == self.editing_original_category:
                original_books[original_index] = book
            else:
                original_books.pop(original_index)
                target_books = self.categories.get(category, [])
                target_books.append(book)
            success_message = "Book updated successfully."
        else:
            book = Book(title, author, isbn, year, quantity)
            try:
                self.add_book(category, book)
            except ValueError as exc:
                messagebox.showerror("Validation Error", str(exc))
                return

            success_message = "Book saved successfully."

        self.refresh_table_from_data()

        if is_editing and self.editing_row_index is not None and self.table is not None:
            max_index = len(self.data) - 1
            selected_index = min(self.editing_row_index, max_index)
            if selected_index >= 0 and hasattr(self.table, "setSelectedRow"):
                self.table.setSelectedRow(selected_index)
                self.table.redraw()

        messagebox.showinfo("Success", success_message)
        self.clear_edit_form()

    def edit_selected_row(self) -> None:
        """Load the selected table row into form fields for editing."""

        if self.table is None or self.data is None or len(self.data) == 0:
            messagebox.showwarning("Selection Required", "No rows available to edit.")
            return

        row_index = self.table.getSelectedRow()
        if row_index is None or row_index < 0 or row_index >= len(self.data):
            messagebox.showwarning("Selection Required", "Select one row in the table to edit.")
            return

        row = self.data.iloc[row_index]
        title = "" if pd.isna(row["title"]) else str(row["title"]).strip()
        author = "" if pd.isna(row["author"]) else str(row["author"]).strip()
        isbn = "" if pd.isna(row["isbn"]) else str(row["isbn"]).strip()
        year = "" if pd.isna(row["year"]) else str(row["year"]).strip()
        quantity_raw = "" if pd.isna(row["quantity"]) else str(row["quantity"]).strip()
        category = "" if pd.isna(row["category"]) else str(row["category"]).strip()

        if not isbn or not category:
            messagebox.showwarning("Invalid Row", "Selected row has no valid ISBN/category.")
            return

        try:
            quantity = int(float(quantity_raw))
        except ValueError:
            quantity = 0

        self.edit_title_var.set(title)
        self.edit_author_var.set(author)
        self.edit_isbn_var.set(isbn)
        self.edit_year_var.set(year)
        self.edit_quantity_var.set(quantity)
        self.edit_category_var.set(category)
        self.editing_original_isbn = isbn
        self.editing_original_category = category
        self.editing_row_index = row_index

    def refresh_table_from_data(self) -> None:
        """Refresh the table model from the current full categorie's dataset."""
        self.set_data(self.categories_to_dataframe())

    def apply_filters(self) -> None:
        """Apply UI filters and update the table with filtered results."""

        try:
            filtered_categories = self.filter_categories(
                category=self.filter_category_var.get(),
                title=self.filter_title_var.get(),
                author=self.filter_author_var.get()
            )
        except ValueError as exc:
            messagebox.showerror("Filter Error", str(exc))
            return

        self.set_data(self.categories_to_dataframe(filtered_categories))

    def clear_filters(self) -> None:
        """Clear filter fields and restore full data in the table."""

        self.filter_title_var.set("")
        self.filter_author_var.set("")
        self.filter_category_var.set("All")
        self.refresh_table_from_data()

    def build_filter(self) -> None:
        """Build filter controls and filter action buttons."""

        filter_frame = tk.LabelFrame(master=self.left_panel_container, text="Filters")
        filter_frame.grid(row=2, column=0, sticky="nwe", padx=10, pady=(0, 0))

        ttk.Label(master=filter_frame, text="Author").grid(row=0, column=0, sticky=tk.W, padx=(10, 10), pady=(0, 10))
        ttk.Entry(master=filter_frame, textvariable=self.filter_author_var).grid(row=0, column=1, sticky="ew", pady=(0, 10), padx=(0, 10))

        ttk.Label(master=filter_frame, text="Title").grid(row=1, column=0, sticky=tk.W, padx=(10, 10), pady=(0, 10))
        ttk.Entry(master=filter_frame, textvariable=self.filter_title_var).grid(row=1, column=1, sticky="ew", pady=(0, 10), padx=(0, 10))

        ttk.Label(master=filter_frame, text="Category").grid(row=2, column=0, sticky=tk.W, padx=(10, 10))
        ttk.Combobox(
            master=filter_frame,
            textvariable=self.filter_category_var,
            values=["All"] + list(self.categories.keys()),
            state="readonly"
        ).grid(row=2, column=1, sticky="ew", pady=(0, 10), padx=(0, 10))

        actions = ttk.Frame(master=filter_frame)
        actions.grid(row=3, column=0, columnspan=2, sticky="ew", padx=10, pady=(0, 10))

        ttk.Button(master=actions, text="Apply Filter", command=self.apply_filters).pack(
            side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 4)
        )
        ttk.Button(master=actions, text="Clear Filter", command=self.clear_filters).pack(
            side=tk.LEFT, fill=tk.X, expand=True, padx=(4, 0)
        )

        filter_frame.grid_columnconfigure(1, weight=1)

    def build_form(self) -> None:
        """Build the create/update form and associated action buttons."""

        form_frame = tk.Frame(master=self.left_panel_container)
        form_frame.grid(row=0, column=0, rowspan=2, sticky="nwe", padx=10, pady=6)

        isbn_validator = (self.register(BookManagement.validate_isbn_field), "%P")
        quantity_validator = (self.register(BookManagement.validate_quantity_field), "%P")

        ttk.Label(master=form_frame, text="Edit Book", style="h9.TLabel").pack(anchor=tk.W)

        sep = tk.Frame(master=form_frame, height=1)
        sep.pack(fill=tk.X, pady=(4, 12))
        sep.configure(background="#444444")

        ttk.Label(master=form_frame, text="Title").pack(anchor=tk.W)
        ttk.Entry(master=form_frame, textvariable=self.edit_title_var).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Author").pack(anchor=tk.W)
        ttk.Entry(master=form_frame, textvariable=self.edit_author_var).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="ISBN").pack(anchor=tk.W)
        ttk.Entry(
            master=form_frame,
            textvariable=self.edit_isbn_var,
            validate="key",
            validatecommand=isbn_validator
        ).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Year").pack(anchor=tk.W)
        ttk.Entry(master=form_frame, textvariable=self.edit_year_var).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Quantity").pack(anchor=tk.W)
        ttk.Spinbox(
            master=form_frame,
            from_=0,
            to=1000,
            textvariable=self.edit_quantity_var,
            validate="key",
            validatecommand=quantity_validator
        ).pack(fill=tk.X, pady=(2, 8))

        ttk.Label(master=form_frame, text="Category").pack(anchor=tk.W)
        ttk.Combobox(
            master=form_frame,
            textvariable=self.edit_category_var,
            values=list(self.categories.keys()),
            state="readonly"
        ).pack(fill=tk.X, pady=(2, 12))

        actions = ttk.Frame(master=form_frame)
        actions.pack(fill=tk.X)

        ttk.Button(master=actions, text="Save Changes", bootstyle="success", command=self.save_changes).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(0, 4))
        ttk.Button(master=actions, text="Clear", bootstyle="secondary", command=self.clear_edit_form).pack(side=tk.LEFT, fill=tk.X, expand=True, padx=(4, 0))

    def save_to_sqlite(self, db_path: str = "library.db") -> None:
        """Persist in-memory categories and books to a SQLite database.

        Args:
            db_path: Destination SQLite file path. Defaults to "library.db".
        """
        db_exists = os.path.exists(db_path)

        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()

            categories_exists = False
            books_exists = False

            if db_exists:
                cursor.execute(
                    "SELECT name FROM sqlite_master WHERE type='table' AND name='categories'"
                )
                categories_exists = cursor.fetchone() is not None

                cursor.execute(
                    "SELECT name FROM sqlite_master WHERE type='table' AND name='books'"
                )
                books_exists = cursor.fetchone() is not None

            if not categories_exists:
                cursor.execute("""
                    CREATE TABLE categories (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        name TEXT NOT NULL UNIQUE
                    )
                """)

            if not books_exists:
                cursor.execute("""
                    CREATE TABLE books (
                        id INTEGER PRIMARY KEY AUTOINCREMENT,
                        title TEXT NOT NULL,
                        author TEXT NOT NULL,
                        isbn TEXT NOT NULL UNIQUE,
                        year INTEGER,
                        quantity INTEGER NOT NULL DEFAULT 0,
                        category_id INTEGER,
                        FOREIGN KEY (category_id) REFERENCES categories(id)
                    )
                """)

            cursor.execute("PRAGMA table_info(books)")
            book_columns = [col[1] for col in cursor.fetchall()]
            if "quantity" not in book_columns:
                cursor.execute("ALTER TABLE books ADD COLUMN quantity INTEGER NOT NULL DEFAULT 0")

            # Full refresh of persisted data based on current in-memory state.
            cursor.execute("DELETE FROM books")
            cursor.execute("DELETE FROM categories")

            for category_name in sorted(self.categories.keys()):
                cursor.execute("INSERT INTO categories (name) VALUES (?)", (category_name,))

            cursor.execute("SELECT id, name FROM categories")
            category_map = {name: category_id for category_id, name in cursor.fetchall()}

            for category_name, category_books in self.categories.items():
                category_id = category_map.get(category_name)
                for book in category_books:
                    if not isinstance(book, Book):
                        continue

                    title = book.title
                    author = book.author
                    isbn = book.isbn
                    year = book.year
                    quantity = int(book.quantity)

                    if not title or not author or not isbn:
                        continue

                    cursor.execute(
                        """
                        INSERT INTO books (title, author, isbn, year, quantity, category_id)
                        VALUES (?, ?, ?, ?, ?, ?)
                        """,
                        (title, author, isbn, year, quantity, category_id)
                    )

            conn.commit()

    def load_data(self, db_path: str = "library.db") -> None:
        """Load categories/books from SQLite and refresh the UI dataset.

        Args:
            db_path: Source SQLite file path. Defaults to "library.db".
        """
        proceed = messagebox.askokcancel(
            "Load Data",
            "Current data will be lost during the data load process. Do you want to continue?"
        )
        if not proceed:
            return

        default_categories = self._default_categories()

        if not os.path.exists(db_path):
            self.categories.clear()
            self.categories.update(default_categories)
            return

        with sqlite3.connect(db_path) as conn:
            cursor = conn.cursor()

            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='categories'")
            categories_exists = cursor.fetchone() is not None
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name='books'")
            books_exists = cursor.fetchone() is not None

            if not categories_exists or not books_exists:
                self.categories.clear()
                self.categories.update(default_categories)
                return

            cursor.execute("""
                SELECT c.name, b.title, b.author, b.isbn, b.year, b.quantity
                FROM categories c
                LEFT JOIN books b ON b.category_id = c.id
                ORDER BY c.name, b.title
            """)
            rows = cursor.fetchall()

        loaded_categories = self._default_categories()
        for category_name, title, author, isbn, year, quantity in rows:
            if category_name not in loaded_categories:
                loaded_categories[category_name] = []

            if not title or not author or not isbn:
                continue

            loaded_categories[category_name].append(Book(title, author, isbn, year, quantity))

        self.categories.clear()
        self.categories.update(loaded_categories)
        self.refresh_table_from_data()

    def categories_to_dataframe(self, categories: dict = None) -> pd.DataFrame:
        """Convert category-book mappings into a table-ready DataFrame.

        Args:
            categories: Optional source mapping. Uses self.categories when omitted.

        Returns:
            pd.DataFrame: Flattened rows with |title|author|isbn|year|quantity|category|.
        """
        source_categories = self.categories if categories is None else categories
        rows = []
        for category_name, books in source_categories.items():
            for book in books:
                if not isinstance(book, Book):
                    continue

                rows.append({
                    "title": book.title,
                    "author": book.author,
                    "isbn": book.isbn,
                    "year": int(book.year),
                    "quantity": book.quantity,
                    "category": category_name
                })
        return pd.DataFrame(rows, columns=["title", "author", "isbn", "year", "quantity", "category"])


## Unit Test

In [59]:
import unittest
from pprint import pprint

class BookManagementTest(unittest.TestCase):
    """Test book management behavior, filtering, and validation rules."""

    def test_get_title(self):
        """Verify that a Book object stores and exposes its title."""
        book = Book("Book 1", "Author 1", "1234567890", 2020, 5)
        print("\nTest Get Title:")
        pprint(book)
        self.assertEqual(book.title, "Book 1")

    def test_add_book(self):
        """Verify add_book inserts the book into the expected category list."""
        book = Book("The Wedding People: A Novel", " Alison Espach", "978-1250899576", 2024, 2)
        management = BookManagement(master=None, no_ui=True)
        management.add_book("Fiction", book)
        self.assertEqual(management.categories["Fiction"], [book])

        print("\nTest Add Book:")
        pprint(management.categories)

    @staticmethod
    def _build_sample_management():
        """Create a reusable in-memory dataset used by filter/duplicate tests."""
        management = BookManagement(master=None, no_ui=True)
        management.add_book("Fiction", Book("Dune", "Frank Herbert", "9780441172719", 1965, 4))
        management.add_book("Science Fiction", Book("Dune Messiah", "Frank Herbert", "9780441172696", 1969, 3))
        management.add_book("Science Fiction", Book("Children of Dune", "Frank Herbert", "9780441104024", 1976, 2))
        management.add_book("Fiction", Book("The Hobbit", "J.R.R. Tolkien", "9780547928227", 1937, 3))
        management.add_book("Mystery", Book("Gone Girl", "Gillian Flynn", "9780307588371", 2012, 2))
        management.add_book("Technology", Book("Clean Code", "Robert C. Martin", "9780132350884", 2008, 5))
        management.add_book("Technology", Book("Clean Architecture", "Robert C. Martin", "9780134494166", 2017, 4))
        return management

    def test_filter_book_by_title(self):
        """Verify title filtering returns all rows containing the given term."""
        management = self._build_sample_management()
        filtered = management.filter_categories(title="dune")

        total_books = sum(len(books) for books in filtered.values())
        self.assertEqual(total_books, 3)
        self.assertEqual(filtered["Fiction"][0].title, "Dune")

        print("\nTest Filter Book by Title:")
        pprint(filtered)

    def test_filter_book_by_author(self):
        """Verify author filtering returns only rows matching the author term."""
        management = self._build_sample_management()
        filtered = management.filter_categories(author="frank")

        total_books = sum(len(books) for books in filtered.values())
        self.assertEqual(total_books, 3)
        self.assertEqual(len(filtered["Science Fiction"]), 2)

        print("\nTest Filter Book by Author:")
        pprint(filtered)

    def test_filter_book_by_category(self):
        """Verify category filtering limits output to one category key."""
        management = self._build_sample_management()
        filtered = management.filter_categories(category="Fiction")

        self.assertEqual(list(filtered.keys()), ["Fiction"])
        self.assertEqual(len(filtered["Fiction"]), 2)

        print("\nTest Filter Book by Category:")
        pprint(filtered)

    def test_filter_book_by_category_author_and_title(self):
        """Verify combined category, title, and author filters work together."""
        management = self._build_sample_management()
        filtered = management.filter_categories(category="Science Fiction", title="dune", author="frank")

        self.assertEqual(list(filtered.keys()), ["Science Fiction"])
        self.assertEqual(len(filtered["Science Fiction"]), 2)
        self.assertEqual(filtered["Science Fiction"][0].author, "Frank Herbert")

        print("\nTest Filter Book by Category, Author, and Title:")
        pprint(filtered)

    def test_has_duplicate_isbn(self):
        """Verify duplicate ISBN detection with and without excluded book context."""
        management = self._build_sample_management()
        self.assertTrue(management.has_duplicate_isbn("9780441172719"))
        self.assertFalse(management.has_duplicate_isbn("9999999999999"))

        current_book = management.categories["Fiction"][0]
        self.assertFalse(management.has_duplicate_isbn("9780441172719", exclude_book=current_book))

    def test_is_valid_isbn_code(self):
        """Verify ISBN format validation accepts valid and rejects invalid values."""
        self.assertTrue(BookManagement.is_valid_isbn_code("1234567890"))
        self.assertTrue(BookManagement.is_valid_isbn_code("1234567890x"))
        self.assertTrue(BookManagement.is_valid_isbn_code("978-1250899576"))
        self.assertFalse(BookManagement.is_valid_isbn_code("123456789023423423423432"))
        self.assertFalse(BookManagement.is_valid_isbn_code("12A0B"))


unittest.main(argv=[''], exit=False)

........
----------------------------------------------------------------------
Ran 8 tests in 0.005s

OK



Test Add Book:
{'Business': [],
 'Education': [],
 'Fantasy': [],
 'Fiction': [Book(title='The Wedding People: A Novel', author=' Alison Espach', isbn='978-1250899576', year=2024, quantity=2)],
 'Mystery': [],
 'Romance': [],
 'Science Fiction': [],
 'Technology': []}

Test Filter Book by Author:
{'Business': [],
 'Education': [],
 'Fantasy': [],
 'Fiction': [Book(title='Dune', author='Frank Herbert', isbn='9780441172719', year=1965, quantity=4)],
 'Mystery': [],
 'Romance': [],
 'Science Fiction': [Book(title='Dune Messiah', author='Frank Herbert', isbn='9780441172696', year=1969, quantity=3),
                     Book(title='Children of Dune', author='Frank Herbert', isbn='9780441104024', year=1976, quantity=2)],
 'Technology': []}

Test Filter Book by Category:
{'Fiction': [Book(title='Dune', author='Frank Herbert', isbn='9780441172719', year=1965, quantity=4),
             Book(title='The Hobbit', author='J.R.R. Tolkien', isbn='9780547928227', year=1937, quantity=3)]}

Test Filter